Notes: (what we did in this project)
- Encoded the categorical labels into a numerical target variable containing 0 if benign, 1 if malignant
- Computed mutual information between the 30 features and the target variable as preliminary analysis of feature importance
- Created a heatmap showing correlation between all variables: the target variable is labeled 'M' in the last row
- Split the dataset into training and testing sets with 70:30 ratio
- Scaled the training and test sets using StandardScaler
- Implemented PCA to reduce the number of features from 30 to 10, with negligible performance difference
- Created a PCA Biplot to visualize the relationships between the first and second principal components and the training dataset
- Implemented KNearestNeighbors, Logistic Regression, Linear Support Vector Machines, Kernel Support Vector Machines, Decision Tree, and an Ensemble Meta-model combining all the previous models
- All models were implemented with hyperparameter optimization through Grid Search Cross-Validation
- The results were plotted as Confusion Matrices, and performance metrics were computed as a table for each model

In [ ]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

# Read the dataset into a pandas DataFrame!
df = pd.read_csv('/kaggle/input/breast-cancer-dataset/Breast_cancer_dataset.csv')

In [ ]:
# Separate the feature and target variables
columns_to_drop = ['id', 'diagnosis', 'Unnamed: 32']
X = df.drop(columns_to_drop, axis=1)
y = df['diagnosis']

# Convert categorical target variable to binary dummy variables
y_num = pd.get_dummies(y, dtype=int)
# Since a sample can only be either 'B' or 'M' and never both, we can discard a column to reduce dimensionality
y_num_ = y_num.drop('B', axis=1)

# Convert the DataFrame into a 1D NumPy array
y_num = y_num_.values.ravel()

In [ ]:
X.shape

In [ ]:
# check mutual information between each feature and Prop_OSY15
from sklearn.feature_selection import mutual_info_regression

mi = mutual_info_regression(X, y_num, random_state=0)
mi_series = pd.Series(mi, index=X.columns)
print(mi_series.sort_values(ascending=False))

In [ ]:
# check the correlation between all variables
allVariables = X.join(y_num_)
allVariables.head()
corr = allVariables.corr()
plt.figure(figsize=(20,20))
sns.heatmap(corr, annot=True, cmap='coolwarm')

plt.show()

In [ ]:
# importing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# initialize
scaler = StandardScaler()

# splitting the train and testing data 70:30
X_train, X_test, y_train, y_test = train_test_split(X, y_num, test_size=0.3, stratify=y_num, random_state=1)

# preprocessing the training and test data
X_train_preprocessed_needpca = scaler.fit_transform(X_train)
X_test_preprocessed_needpca = scaler.transform(X_test)

In [ ]:
from sklearn.decomposition import PCA

# initialize PCA & keep enough components to explain 95% of the variance
pca = PCA(n_components=0.95)

# fit PCA on training set
X_train_preprocessed = pca.fit_transform(X_train_preprocessed_needpca)

# transform testing set
X_test_preprocessed = pca.transform(X_test_preprocessed_needpca)

# check results
print("Original number of features:", X_train_preprocessed_needpca.shape[1])
print("Number of features after PCA:", X_train_preprocessed.shape[1])
print("Explained variance ratio of each component:", pca.explained_variance_ratio_)
print("Cumulative explained variance for n components:", pca.explained_variance_ratio_.cumsum())


Using PCA, we reduced our feature count from 30 features to just 10. 

I have also checked prior that the results of not implementing PCA vs implementing PCA is negligible, with accuracy improving by <1%.

This means our feature reduction is effective.

In [ ]:
from matplotlib.colors import ListedColormap

# initialize labels and colors
class_labels = ['Benign', 'Malignant']
colors_list = ['blue', 'red']
cmap = ListedColormap(colors_list)

# Map each sample to a color index
y_colors = y_train  # should be 0 or 1

# PCA biplot
xs = X_train_preprocessed[:, 0]
ys = X_train_preprocessed[:, 1]

plt.figure(figsize=(10, 8))
scatter = plt.scatter(xs, ys, c=y_colors, cmap=cmap, alpha=0.8, s=60)

# Colorbar with class names
cbar = plt.colorbar(scatter, ticks=[0, 1])  # center ticks for 0 and 1
cbar.set_ticklabels(class_labels)
cbar.set_label("Class")

# Plot feature loadings
loadings = pca.components_.T
for i, feature in enumerate(X.columns):
    plt.arrow(0, 0,
              loadings[i, 0] * max(xs),
              loadings[i, 1] * max(ys),
              color='black', alpha=0.8, head_width=0.05)
    plt.text(loadings[i, 0] * max(xs) * 1.1,
             loadings[i, 1] * max(ys) * 1.1,
             feature, color='black', ha='center', va='center')

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("PCA Biplot (PC1 vs PC2)")
plt.axhline(0, color='grey', lw=1)
plt.axvline(0, color='grey', lw=1)
plt.grid(True)
plt.show()


In [ ]:
# Import some more goodz
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, KFold

# Performing GridSearchCV to optimize the hyperparameter
splits = 5 # this is arbitrarily chosen
kf = KFold(n_splits=splits, shuffle=True, random_state=1)
knn_param_grid = {'n_neighbors': range(1,26)}
knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, knn_param_grid, cv=kf)
knn_cv.fit(X_train_preprocessed, y_train)
print(knn_cv.best_params_, knn_cv.best_score_)

We choose n_neighbors = 3!

In [ ]:
# Visualizing the Hyperparameter Tuning
import matplotlib.pyplot as plt

knn_results_df = knn_cv.cv_results_['mean_test_score']
n_neighbors = knn_param_grid['n_neighbors']

plt.figure(figsize=(6,5))
plt.plot(n_neighbors, knn_results_df, marker='o')
plt.xlabel('Number of neighbors')
plt.ylabel('Cross-validated accuracy')
plt.title('KNN Hyperparameter Tuning using GridSearchCV')
plt.xticks(n_neighbors)
plt.show()

In [ ]:
# Construct the confusion matrix and classification report to evaluate our model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

knn_y_pred = knn_cv.predict(X_test_preprocessed)

conf_matrix = confusion_matrix(y_test, knn_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Benign', 'Malignant'])
disp.plot(cmap=plt.cm.Blues)
plt.title('KNeighbors Confusion Matrix')
plt.show()
print(classification_report(y_test, knn_y_pred, digits=4))

We have 7 cases of False Negatives, meaning our model predicted the tumor was benign, when it was actually malignant. The accuracy is 95.32%.

This is unacceptable! Let's try Logistic Regression this time.

In [ ]:
# We will now try Logistic Regression and see if it yields better results
from sklearn.linear_model import LogisticRegression

# Performing GridSearchCV to find optimize the hyperparameters 
# (strength of regularization (C) and regularization techniques (l1 or l2))
lr_param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                'penalty': ['l1', 'l2'], 
                'max_iter' : [100, 200, 500, 1000]}
lr = LogisticRegression(solver='liblinear')
lr_cv = GridSearchCV(lr, lr_param_grid, cv=kf)
lr_cv.fit(X_train_preprocessed, y_train)
print(lr_cv.best_params_, lr_cv.best_score_)

We choose C = 1, maximum iterations = 100, and regularization technique = l2 (Ridge regression)!

In [ ]:
# Visualizing the hyperparameter tuning for Logistic Regression
import seaborn as sns
# Seaborn was giving me FutureWarnings so I'm just doing this as a bandaid solution
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Load the results into a DataFrame
lr_results_df = pd.DataFrame(lr_cv.cv_results_)

sns.lineplot(data=lr_results_df, x='param_C', y='mean_test_score', hue='param_penalty', marker='o')
plt.xscale('log')
plt.xlabel('Regularization Strength (value of C)')
plt.ylabel('Mean Test Score')
plt.grid(True)
plt.title('Logistic Regression Hyperparameter Tuning')
plt.show()

In [ ]:
# Now let's use our model with the optimized hyperparameters and analyze the results!
lr_y_pred = lr_cv.predict(X_test_preprocessed)

conf_matrix = confusion_matrix(y_test, lr_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Benign', 'Malignant'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Logistic Regression Confusion Matrix')
plt.show()
print(classification_report(y_test, lr_y_pred, digits=4))

We have achieved better results with logistic regression compared to k-neighbors classification.

But there are 5 false negatives-- which is a slight improvement! The accuracy is 96.49%.

I wonder-- can we achieve better results? 

I have to study more. Maybe better feature engineering or hyperparameter tuning?

Now, let's try using Linear SVM, and see if it yields better results

In [ ]:
from sklearn.svm import LinearSVC
# Instantiate the Linear SVM and perform cross-validation to tune the hyperparameters
linsvm_param_grid = {'C': [0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.1, 1, 2]}
linsvm = LinearSVC()
linsvm_cv = GridSearchCV(linsvm, linsvm_param_grid, cv=kf)
linsvm_cv.fit(X_train_preprocessed, y_train)
print(linsvm_cv.best_params_, linsvm_cv.best_score_)

In [ ]:
# Load the results into a DataFrame
linsvm_results_df = pd.DataFrame(linsvm_cv.cv_results_)

sns.lineplot(data=linsvm_results_df, x='param_C', y='mean_test_score', marker='o')
plt.xscale('log')
plt.xlabel('Regularization Strength (value of C)')
plt.ylabel('Mean Test Score')
plt.grid(True)
plt.title('Linear SVM Hyperparameter Tuning')
plt.show()

In [ ]:
# Now let's use our model with the optimized hyperparameters and analyze the results!
linsvm_y_pred = linsvm_cv.predict(X_test_preprocessed)

conf_matrix = confusion_matrix(y_test, linsvm_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Benign', 'Malignant'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Linear SVM Confusion Matrix')
plt.show()
print(classification_report(y_test, linsvm_y_pred, digits=4))

Now, there are 6 False Negatives. The accuracy is 95.91%.


This is the best we can do so far. I will have to study more to get better results!

Now, let's try the SVM using kernel radial basis function or RBF.

In [ ]:
from sklearn.svm import SVC
# Instantiate the SVC and perform cross validation to tune the hyperparameters
svm = SVC()
svm_param_grid = {'C': np.linspace(0.001, 5.0, 50),
                 'gamma': [0.0001, 0.001, 0.01, 0.1, 1]}
svm_cv = GridSearchCV(svm, svm_param_grid, cv=kf)

svm_cv.fit(X_train_preprocessed, y_train)

print(svm_cv.best_params_, svm_cv.best_score_)

In [ ]:
# Visualize hyperparameter tuning
svm_cv_results_df = pd.DataFrame(svm_cv.cv_results_)
sns.lineplot(data=svm_cv_results_df, x='param_C', y='mean_test_score', hue='param_gamma', marker='o')

plt.xlabel('Regularization Strength (value of C)')
plt.ylabel('Mean Test Score')
plt.grid(True)
plt.title('Kernel SVM Hyperparameter Tuning')
plt.show()

In [ ]:
# Now let's use our model with the optimized hyperparameters and analyze the results!
svm_y_pred = svm_cv.predict(X_test_preprocessed)

conf_matrix = confusion_matrix(y_test, svm_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Benign', 'Malignant'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Kernel SVM Confusion Matrix')
plt.show()
print(classification_report(y_test, svm_y_pred, digits=4))

There are 6 False Positives. We can observe that the kernel SVM does not perform better in this case. I wonder if it is the dataset, the model itself, or my modeling that is causing these high errors. The accuracy is 95.91%.

In [ ]:
#Instantiate the Decision Tree Classifier and perform cross-validation to tune hyperparameters while avoiding overfitting
from sklearn.tree import DecisionTreeClassifier

dt_param_grid = {'criterion': ['gini', 'entropy'],
                'max_depth': np.arange(1, 21)}
dt = DecisionTreeClassifier()
dt_cv = GridSearchCV(dt, param_grid=dt_param_grid, cv=kf)

dt_cv.fit(X_train_preprocessed, y_train)
print(dt_cv.best_params_, dt_cv.best_score_)


In [ ]:
# Visualize hyperparameter tuning of the Decision Tree
dt_cv_results_df = pd.DataFrame(dt_cv.cv_results_)
sns.lineplot(data=dt_cv_results_df, x='param_max_depth', y='mean_test_score', hue='param_criterion', marker='o')

plt.xlabel('Maximum Depth')
plt.xticks(np.arange(1,21))
plt.ylabel('Mean Test Score')
plt.grid(True)
plt.title('Decision Tree Hyperparameter Tuning')
plt.show()

In [ ]:
# Now let's use our model with the optimized hyperparameters and analyze the results!
dt_y_pred = dt_cv.predict(X_test_preprocessed)

conf_matrix = confusion_matrix(y_test, dt_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Benign', 'Malignant'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Decision Tree Confusion Matrix')
plt.show()
print(classification_report(y_test, dt_y_pred, digits=4))

The Decision Tree Classifier also does not do so well in this case, resulting in 9 False Negatives. The accuracy is 90.64%. 

In [ ]:
# Now let's try Ensemble and combine all our previous models!

from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

classifiers = [('KNeighborsClassifier', knn_cv), 
               ('Logistic Regression', lr_cv), 
               ('SVM', svm_cv), 
               ('DecisionTreeClassifier', dt_cv)]

vc = VotingClassifier(estimators=classifiers)

vc.fit(X_train_preprocessed, y_train)

vc_y_pred = vc.predict(X_test_preprocessed)

vc_accuracy = accuracy_score(y_test, vc_y_pred)

print(vc_accuracy)

In [ ]:
# Visualizing the results

conf_matrix = confusion_matrix(y_test, vc_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Benign', 'Malignant'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Ensemble Meta-model Confusion Matrix')
plt.show()
print(classification_report(y_test, vc_y_pred, digits=4))

The accuracy is 95.91%. We can see that the Ensemble Meta-Model does not perform better than the Logistic Regression model in this case.

# Thank you for going on this journey with me! -Clyde

## 📥 Download Trained Model & Artifacts (Kaggle)

This cell saves the trained classification model (`breast_cancer_model.pkl`), ensemble model (`breast_cancer_ensemble.pkl`), `StandardScaler` (`breast_cancer_scaler.pkl`), and `PCA` (`breast_cancer_pca.pkl`), packages them into a single zip archive, and creates direct download links.

In [ ]:
# ====================================================================
# 📥 SAVE & DOWNLOAD TRAINED ARTIFACTS ON KAGGLE
# ====================================================================
import os
import pickle
import zipfile
from IPython.display import FileLink, display

saved_files = []

# 1. Save Best Performing Model (Logistic Regression)
if 'lr_cv' in globals():
    model_path = 'breast_cancer_model.pkl'
    best_estimator = getattr(lr_cv, 'best_estimator_', lr_cv)
    with open(model_path, 'wb') as f:
        pickle.dump(best_estimator, f)
    saved_files.append(model_path)
    print(f"✓ Saved Logistic Regression model: {model_path}")

# 2. Save Ensemble Model if available
if 'vc' in globals():
    ens_path = 'breast_cancer_ensemble.pkl'
    with open(ens_path, 'wb') as f:
        pickle.dump(vc, f)
    saved_files.append(ens_path)
    print(f"✓ Saved Ensemble model: {ens_path}")

# 3. Save StandardScaler
if 'scaler' in globals():
    scaler_path = 'breast_cancer_scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    saved_files.append(scaler_path)
    print(f"✓ Saved StandardScaler: {scaler_path}")

# 4. Save PCA
if 'pca' in globals():
    pca_path = 'breast_cancer_pca.pkl'
    with open(pca_path, 'wb') as f:
        pickle.dump(pca, f)
    saved_files.append(pca_path)
    print(f"✓ Saved PCA: {pca_path}")

# 5. Package all files into a single ZIP archive for 1-click download
zip_filename = 'breast_cancer_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file_name in saved_files:
        if os.path.exists(file_name):
            zipf.write(file_name, os.path.basename(file_name))
print(f"✓ Packaged bundle: {zip_filename}")

# 6. Generate Kaggle Direct Download Links
print("\n" + "="*50)
print("👉 Click the link below to download your complete model bundle:")
print("="*50)
display(FileLink(zip_filename))

print("\nIndividual file downloads:")
for file_name in saved_files:
    if os.path.exists(file_name):
        display(FileLink(file_name))
